In [21]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import warnings
import matplotlib
import matplotlib.pyplot as plt
import sys 
import os 
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np
import seaborn as sns
from tqdm import tqdm
import subprocess
from sklearn.metrics import r2_score

import pandas as pd
from scipy.stats import linregress
import matplotlib.patches as mpatches
from scipy.stats import pearsonr, spearmanr
from pandas.api.types import CategoricalDtype
from scipy.cluster.hierarchy import linkage
from matplotlib.patches import Patch
from statsmodels.stats.multitest import multipletests

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)
# np.set_printoptions(threshold=np.inf)

plt.rcParams["figure.figsize"]=4,4

warnings.filterwarnings("ignore")

# - dirs 
from ciim.src.common import PLOTS_DIR, base_dir, aging_clock_train_datasets, palette_genders, palette_treatment, \
        mapping_major_2_minor, mapping_minor_2_major, cell_types, datasets_e, datasets_all, datasets_a, palette_datasets, \
        datasets_disease, datasets_drug_perturbation, SAVE_DIR, \
        palette_datasets_pretty, surrogate_names, colors_blind, palette_trend, palette_trend_2, palette_regulation, palette_cell_types


In [85]:
!ls ../base_folder/output/perturbations/

rejuvenating_drugs_CXCL9.csv           reversal_stats_CXCL9.csv
rejuvenating_drugs_op.csv              reversal_stats_op.csv
rejuvenating_drugs_parsebioscience.csv reversal_stats_parsebioscience.csv


In [84]:
!cd ../ && bash  scripts/experiment/run_reversal_pipeline.sh op "CD4T CD8T" tf_activity 2>&1 
!cd ../ && bash scripts/experiment/run_reversal_pipeline.sh CXCL9 "CD4T CD8T" tf_activity 2>&1 
!cd ../ && bash scripts/experiment/run_reversal_pipeline.sh parsebioscience "CD4T CD8T" tf_activity 2>&1 

DRUG REVERSAL ANALYSIS PIPELINE

Configuration:
  Dataset: op
  Cell types: CD4T CD8T
  Feature type: tf_activity


STEP 1: Computing drug statistics for all TFs...

STEP 2: Running reversal analysis...
Drug Reversal Analysis
Dataset: op
Cell types: CD4T CD8T
Feature type: tf_activity


DRUG REVERSAL ANALYSIS

Dataset: op
Feature type: tf_activity
Cell types: ['CD4T', 'CD8T']
Significance threshold: 0.05
Use all drug stats: True

Loading aging statistics...
  Loaded 431 aging TF associations
  Cell types: ['CD4T', 'CD8T', 'NK', 'B', 'MONO']
  Unique TFs: 298

Loading drug statistics...
  Loaded 43543 drug TF associations
  Unique drugs/comparisons: 134
  Drugs: ['TIE2 Kinase Inhibitor', 'MK-5108', 'Lapatinib', 'Belinostat', 'Dabrafenib', 'Atorvastatin', 'Clomipramine', 'Sunitinib', 'Tivozanib', 'Tamatinib', 'Ixabepilone', 'Linagliptin', '5-(9-Isopropyl-8-methyl-2-morpholino-9H-purin-6-yl)pyrimidin-2-amine', 'O-Demethylated Adapalene', 'Protriptyline', 'I-BET151', 'Foretinib', 'GW843682

In [ ]:
!ls -lt {base_dir}/output/perturbations/rejuvenating_drugs_parsebioscience.csv

total 128
-rw-r--r--@ 1 jno24  1382859752    596 Nov 30 14:36 rejuvenating_drugs_parsebioscience.csv
-rw-r--r--@ 1 jno24  1382859752  23418 Nov 30 14:36 reversal_stats_parsebioscience.csv
-rw-r--r--@ 1 jno24  1382859752    633 Nov 29 19:03 rejuvenating_drugs_CXCL9.csv
-rw-r--r--@ 1 jno24  1382859752    735 Nov 29 19:03 reversal_stats_CXCL9.csv
-rw-r--r--@ 1 jno24  1382859752   6335 Nov 28 21:12 rejuvenating_drugs_op.csv
-rw-r--r--@ 1 jno24  1382859752  19418 Nov 28 21:12 reversal_stats_op.csv


In [100]:
dataset = 'parsebioscience'  #'CXCL9' # 'op' parsebioscience
df_all = pd.read_csv(f'{base_dir}/output/perturbations/reversal_stats_{dataset}.csv')
df_all[(df_all['cell_type']=='CD4T') & (df_all['drug']=='IL-10')]

,drug,cell_type,dataset,n_reversal,n_decrease_increasing,n_increase_decreasing,n_acceleration,n_common,fisher_pvalue,odds_ratio,reversal_score,effect_type,a,b,c,d,fisher_pvalue_adj,classification
17,IL-10,CD4T,parsebioscience,22,5,17,18,40,0.060542,0.0,0.1,reversal,18,5,17,0,0.096438,neutral


In [99]:
# Check what files were generated
dataset = 'parsebioscience'
df_all = pd.read_csv(f'{base_dir}/output/perturbations/reversal_stats_{dataset}.csv')
df = pd.read_csv(f'{base_dir}/output/perturbations/rejuvenating_drugs_{dataset}.csv')

print('Rejuvinative drugs in CD4T: ', 'number: ', df[df['cell_type']=='CD4T']['drug'].nunique() , ' list: ', df[df['cell_type']=='CD4T']['drug'].tolist())
df[df['cell_type']=='CD4T']

Rejuvinative drugs in CD4T:  number:  1  list:  ['IL-6']


,drug,cell_type,dataset,n_reversal,n_decrease_increasing,n_increase_decreasing,n_acceleration,n_common,fisher_pvalue,odds_ratio,reversal_score,effect_type,a,b,c,d,fisher_pvalue_adj,classification
2,IL-6,CD4T,parsebioscience,15,12,3,1,16,0.007143,0.0,0.875,reversal,0,12,3,1,0.012605,rejuvenating


In [77]:
aging_clocks_rej_drugs = ['Tamatinib',
 '5-(9-Isopropyl-8-methyl-2-morpholino-9H-purin-6-yl)pyrimidin-2-amine',
 'Foretinib',
 'TL_HRAS26',
 'Ruxolitinib',
 'AVL-292',
 'LY2090314',
 'Dasatinib',
 'PF-04691502',
 'Perhexiline',
 'Flutamide',
 'BMS-536924',
 'IKK Inhibitor VII',
 'Idelalisib',
 'PD-0325901',
 'CHIR-99021',
 'Defactinib',
 'BI-D1870',
 'Crizotinib',
 'Nilotinib',
 'Saracatinib',
 'PRT-062607',
 'MGCD-265',
 'R428',
 'Selumetinib',
 'GLPG0634']
df = pd.read_csv(f'{base_dir}/output/perturbations/rejuvenating_drugs_op.csv')
rej_drugs = df[df['cell_type']=='CD4T']['drug'].tolist()
len(set(rej_drugs).intersection(set(aging_clocks_rej_drugs)))

19

- find the common compounds between aging clocks, fisher's test
- repeat for each dataset of op and clcx9
- repeat for promotor based 
- evaluate for cd8t